# Interpreting Recurrent Models

This notebook demonstrates Captum attribution on an LSTM classifier whose input shape is `(batch, sequence_length, features)`. It shows how to compute input attributions, how to aggregate them by time step or by feature, and how to use layer attribution on the recurrent layer.

**Note:** Before running this tutorial, please install matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from captum.attr import IntegratedGradients, LayerIntegratedGradients

## Create a sequence classification task

The synthetic label depends on feature 0 near the end of the sequence, feature 1 near the beginning, and feature 2 in the middle. This gives us a known pattern to compare against the attribution summaries.

In [ ]:
torch.manual_seed(7)

def make_sequences(num_examples, seq_len=24, num_features=4):
    x = torch.randn(num_examples, seq_len, num_features)
    time_feature = torch.linspace(-1, 1, seq_len).view(1, seq_len)
    x[:, :, 3] = time_feature

    evidence = (
        x[:, -8:, 0].sum(dim=1)
        - x[:, :8, 1].sum(dim=1)
        + 0.5 * x[:, 8:16, 2].sum(dim=1)
    )
    y = (evidence > 0).long()
    return x, y

train_x, train_y = make_sequences(2048)
test_x, test_y = make_sequences(256)

train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=64,
    shuffle=True,
)

## Train a small LSTM classifier

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :])

model = LSTMClassifier(input_size=4, hidden_size=16, output_size=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(8):
    model.train()
    for batch_x, batch_y in train_loader:
        logits = model(batch_x)
        loss = F.cross_entropy(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    accuracy = (model(test_x).argmax(dim=1) == test_y).float().mean()

print("Test accuracy:", accuracy.item())

## Attribute a prediction to sequence inputs

`IntegratedGradients` returns attributions with the same shape as the input. For this model, that means one attribution score per `(time step, feature)` pair.

In [ ]:
example = test_x[:1]
baseline = torch.zeros_like(example)
target = model(example).argmax(dim=1).item()

ig = IntegratedGradients(model)

# CuDNN RNN backward does not support eval mode on some GPU setups.
with torch.backends.cudnn.flags(enabled=False):
    attributions, delta = ig.attribute(
        example,
        baselines=baseline,
        target=target,
        return_convergence_delta=True,
    )

print("Attribution shape:", tuple(attributions.shape))
print("Convergence delta:", delta.item())

Aggregate over the feature dimension to rank time steps, or aggregate over the time dimension to rank input features.

In [ ]:
time_importance = attributions.abs().sum(dim=2).squeeze(0).detach()
feature_importance = attributions.abs().sum(dim=1).squeeze(0).detach()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(example.shape[1]), time_importance)
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Absolute attribution")
axes[0].set_title("Importance by time step")

axes[1].bar(range(example.shape[2]), feature_importance)
axes[1].set_xlabel("Feature")
axes[1].set_title("Importance by feature")

plt.tight_layout()
plt.show()

## Attribute through the recurrent layer

`LayerIntegratedGradients` can attribute to the recurrent layer inputs or outputs. Attributing to layer inputs keeps the result aligned with the original `(sequence_length, features)` axes.

In [ ]:
lig = LayerIntegratedGradients(model, model.lstm)

with torch.backends.cudnn.flags(enabled=False):
    layer_input_attr = lig.attribute(
        example,
        baselines=baseline,
        target=target,
        attribute_to_layer_input=True,
    )

print("Layer input attribution shape:", tuple(layer_input_attr.shape))